# Assignment: Prompting, Structured Outputs, and Evaluation

This assignment has **three independent sections**, each
built on a real, public dataset (loaded live via the `datasets` library):

| Section | Lecture | Topic | Dataset |
|---|---|---|---|
| 1 | L5 / Lec6 | Few-shot prompting for sentiment classification | GLUE SST-2 (`nyu-mll/glue`) |
| 2 | Lec6 | Structured output extraction with Pydantic + validated JSON | Customer Support Tickets (`Tobi-Bueck/customer-support-tickets`) |
| 3 | Lec7 | Evaluation metrics (ROUGE, BLEU) and Cohen's Kappa | CNN/DailyMail (`abisee/cnn_dailymail`) |

**Rules — read before you start:**
- Do **not** rename any variable, function, or class that is already given in a cell. The grading rubric checks these exact names.
- Do **not** change any of the fixed "DO NOT MODIFY" data-loading cells — they select a
  fixed subset of each dataset so every student works on the exact same examples. Only
  fill in the cells marked `# TODO`.
- Run all cells top to bottom (`Runtime > Run all`) before submitting. Submit the `.ipynb`
  file with all outputs visible. The first run of each section will download a small
  dataset from the Hugging Face Hub, this needs an internet connection (Colab has one
  by default).
- Section 1 and Section 2 run a **local, open-source model** — no API key needed. The
  first run will download the model's weights (a few GB) from the Hugging Face Hub, so
  use a GPU runtime for reasonable speed: **Runtime > Change runtime type > T4 GPU**. It
  will also run on CPU, just more slowly. Section 3 needs neither an API key nor a GPU.


## Section 0 — Setup (do not modify)

Run these cells once at the start.


In [1]:
!pip install -q transformers accelerate pydantic rouge_score nltk datasets


  Preparing metadata (setup.py) ... done


In [2]:
import json
from typing import Literal

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from pydantic import BaseModel
from datasets import load_dataset

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu
from nltk.tokenize import word_tokenize


In [3]:
# We use an open, ungated, instruction-tuned model that runs on a free Colab GPU (or
# CPU, just slower). No API key needed. Every student uses the same model, so results
# are comparable.
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading {MODEL_NAME} on {device} ...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
model.eval()

print("Model loaded.")


Loading Qwen/Qwen2.5-1.5B-Instruct on cuda ...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.


## Section 1 — Few-Shot Prompting for Sentiment Classification (17 marks)

**Dataset.** [GLUE SST-2](https://huggingface.co/datasets/nyu-mll/glue) (Stanford
Sentiment Treebank, binary version). Label `0` = Negative, `1` = Positive.

**Fixed subsets (loaded for you in the cell below — do not change them):**
- `EVAL_EXAMPLES`: the first 10 rows of the SST-2 **validation** split.
- `FEW_SHOT_EXAMPLES`: the first 2 **negative**-labeled rows of the **train** split,
  followed by the first 2 **positive**-labeled rows of the **train** split (4 examples
  total, in that fixed order).

**Task.** You must:
1. Build a **zero-shot prompt** for a sentence using the exact template given below.
2. Build a **few-shot prompt** for a sentence that includes all 4 `FEW_SHOT_EXAMPLES`
   (in the fixed order above) before the target sentence, using the exact template given
   below.
3. Call the local model with **greedy decoding** (`do_sample=False` — the open-model
   equivalent of `temperature=0`) and `max_new_tokens=5` to get the model's answer.
4. Compute accuracy for the zero-shot approach and the few-shot approach separately, over
   all 10 rows of `EVAL_EXAMPLES`.

**Do not change:** `EVAL_EXAMPLES`, `FEW_SHOT_EXAMPLES`, `MODEL_NAME`, or any function
name below.

**Exact zero-shot template** (fill in `{sentence}`):
```
Classify the sentiment of the following sentence as exactly one word: Positive or Negative.

Sentence: "{sentence}"
Sentiment:
```

**Exact few-shot template**: the same instruction line, followed by each of the 4
`FEW_SHOT_EXAMPLES` formatted as `Sentence: "..."` / `Sentiment: ...` (blank line between
each), followed by the target sentence in the same format with `Sentiment:` left blank
for the model to complete.


In [4]:
# Fixed dataset subsets — DO NOT MODIFY
# Loads GLUE SST-2 and selects the same fixed rows for every student.
LABEL_NAMES = {0: "Negative", 1: "Positive"}

sst2 = load_dataset("nyu-mll/glue", "sst2")

# First 10 rows of the validation split
EVAL_EXAMPLES = [
    {"sentence": ex["sentence"].strip(), "label": LABEL_NAMES[ex["label"]]}
    for ex in sst2["validation"].select(range(10))
]

# First 2 negative rows, then first 2 positive rows, of the train split
_few_shot_negative = sst2["train"].filter(lambda ex: ex["label"] == 0).select(range(2))
_few_shot_positive = sst2["train"].filter(lambda ex: ex["label"] == 1).select(range(2))
FEW_SHOT_EXAMPLES = (
    [{"sentence": ex["sentence"].strip(), "label": "Negative"} for ex in _few_shot_negative]
    + [{"sentence": ex["sentence"].strip(), "label": "Positive"} for ex in _few_shot_positive]
)

print("EVAL_EXAMPLES:")
for ex in EVAL_EXAMPLES:
    print(f"  [{ex['label']}] {ex['sentence']}")

print("\nFEW_SHOT_EXAMPLES:")
for ex in FEW_SHOT_EXAMPLES:
    print(f"  [{ex['label']}] {ex['sentence']}")


README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

sst2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sst2/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

sst2/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Filter:   0%|          | 0/67349 [00:00<?, ? examples/s]

Filter:   0%|          | 0/67349 [00:00<?, ? examples/s]

EVAL_EXAMPLES:
  [Positive] it 's a charming and often affecting journey .
  [Negative] unflinchingly bleak and desperate
  [Positive] allows us to hope that nolan is poised to embark a major career as a commercial yet inventive filmmaker .
  [Positive] the acting , costumes , music , cinematography and sound are all astounding given the production 's austere locales .
  [Negative] it 's slow -- very , very slow .
  [Positive] although laced with humor and a few fanciful touches , the film is a refreshingly serious look at young women .
  [Negative] a sometimes tedious film .
  [Negative] or doing last year 's taxes with your ex-wife .
  [Positive] you do n't have to know about music to appreciate the film 's easygoing blend of comedy and romance .
  [Negative] in exactly 89 minutes , most of which passed as slowly as if i 'd been sitting naked on an igloo , formula 51 sank from quirky to jerky to utter turkey .

FEW_SHOT_EXAMPLES:
  [Negative] hide new secretions from the parental uni

### TODO 1.1 — `build_zero_shot_prompt` (3 marks)

In [5]:
def build_zero_shot_prompt(sentence):
    """Return the exact zero-shot prompt string for one sentence."""
    return (
        "Classify the sentiment of the following sentence as exactly one word: "
        "Positive or Negative.\n\n"
        f'Sentence: "{sentence}"\n'
        "Sentiment:"
    )

# Quick check
print(build_zero_shot_prompt(EVAL_EXAMPLES[0]["sentence"]))

Classify the sentiment of the following sentence as exactly one word: Positive or Negative.

Sentence: "it 's a charming and often affecting journey ."
Sentiment:


### TODO 1.2 — `build_few_shot_prompt` (5 marks)

In [6]:
def build_few_shot_prompt(sentence):
    """Return the exact few-shot prompt string for one sentence.
    Must include all 4 FEW_SHOT_EXAMPLES, in order, before the target sentence."""
    lines = [
        "Classify the sentiment of the following sentence as exactly one word: "
        "Positive or Negative.",
        ""
    ]

    for ex in FEW_SHOT_EXAMPLES:
        lines.append(f'Sentence: "{ex["sentence"]}"')
        lines.append(f'Sentiment: {ex["label"]}')
        lines.append("")

    lines.append(f'Sentence: "{sentence}"')
    lines.append("Sentiment:")

    return "\n".join(lines)

# Quick check
print(build_few_shot_prompt(EVAL_EXAMPLES[0]["sentence"]))

Classify the sentiment of the following sentence as exactly one word: Positive or Negative.

Sentence: "hide new secretions from the parental units"
Sentiment: Negative

Sentence: "contains no wit , only labored gags"
Sentiment: Negative

Sentence: "that loves its characters and communicates something rather beautiful about human nature"
Sentiment: Positive

Sentence: "demonstrates that the director of such hollywood blockbusters as patriot games can still turn out a small , personal film with an emotional wallop ."
Sentiment: Positive

Sentence: "it 's a charming and often affecting journey ."
Sentiment:


### TODO 1.3 — `get_model_response` and `predict_label` (4 marks)


In [7]:
def get_model_response(prompt, max_new_tokens=5):
    """Call the local model with greedy decoding (do_sample=False — the open-model
    equivalent of temperature=0) and return the newly generated text, stripped."""
    messages = [{"role": "user", "content": prompt}]
    chat_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(chat_prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    new_token_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_token_ids, skip_special_tokens=True)

    return response.strip()


def predict_label(prompt):
    """Call get_model_response and normalize the output to exactly "Positive",
    "Negative", or "Unknown" (if the model did not return one of those two words)."""
    response = get_model_response(prompt)
    first_word = response.strip().split()[0] if response.strip() else ""

    first_word = first_word.strip(".,!?;:'\"()[]{}").lower()

    if first_word == "positive":
        return "Positive"
    if first_word == "negative":
        return "Negative"

    return "Unknown"

### TODO 1.4 — Run both prompting strategies and compute accuracy (5 marks)

In [8]:
true_labels = [ex["label"] for ex in EVAL_EXAMPLES]

# Build predictions using the two different prompt strategies.
zero_shot_predictions = [
    predict_label(build_zero_shot_prompt(ex["sentence"]))
    for ex in EVAL_EXAMPLES
]

few_shot_predictions = [
    predict_label(build_few_shot_prompt(ex["sentence"]))
    for ex in EVAL_EXAMPLES
]

# Accuracy = number of correct predictions divided by the total number of examples.
zero_shot_accuracy = sum(
    pred == true for pred, true in zip(zero_shot_predictions, true_labels)
) / len(true_labels)

few_shot_accuracy = sum(
    pred == true for pred, true in zip(few_shot_predictions, true_labels)
) / len(true_labels)

print("Zero-shot predictions:", zero_shot_predictions)
print("Few-shot predictions: ", few_shot_predictions)
print("True labels:          ", true_labels)
print(f"\nZero-shot accuracy: {zero_shot_accuracy}")
print(f"Few-shot accuracy:  {few_shot_accuracy}")

Zero-shot predictions: ['Positive', 'Negative', 'Positive', 'Positive', 'Negative', 'Positive', 'Negative', 'Negative', 'Positive', 'Negative']
Few-shot predictions:  ['Positive', 'Negative', 'Positive', 'Positive', 'Negative', 'Positive', 'Negative', 'Negative', 'Positive', 'Negative']
True labels:           ['Positive', 'Negative', 'Positive', 'Positive', 'Negative', 'Positive', 'Negative', 'Negative', 'Positive', 'Negative']

Zero-shot accuracy: 1.0
Few-shot accuracy:  1.0


## Section 2 — Structured Extraction with Pydantic + Validated JSON (17 marks)

**Dataset.** [Tobi-Bueck/customer-support-tickets](https://huggingface.co/datasets/Tobi-Bueck/customer-support-tickets)
— a real, multilingual customer-support ticket dataset (CC BY-NC 4.0) with a `queue`
(category) and a `priority` (urgency) field per ticket, matching exactly the two fields
we want to extract.

**Fixed subset (loaded for you in the cell below — do not change it).** We pick 6 real
English tickets by their exact `subject` line, so every student extracts from the same 6
tickets. Their `true_category` and `true_urgency` below come directly from the dataset's
own `queue` and `priority` columns.

**Note on tool calling vs. local models.** OpenAI's API can *force* a model to return
arguments matching a schema (tool calling). Most open-source models running locally via
`transformers` have no such built-in guarantee — so instead we prompt the model with the
JSON schema and instructions to return only JSON, then **validate** the result with
Pydantic, retrying a few times if parsing fails. This is the standard fallback pattern
for structured output with local/open models.

You must:
1. Define a Pydantic model named exactly `TicketExtraction` with exactly two fields:
   - `category`: one of `"Technical Support"`, `"Billing and Payments"`, `"Returns and Exchanges"`
   - `urgency`: one of `"high"`, `"medium"`, `"low"`
2. Write `extract_ticket_info(ticket_text)` that prompts the local model with
   `TicketExtraction.model_json_schema()` embedded in the prompt, asks for a single JSON
   object matching that schema and nothing else, and validates the result with
   `TicketExtraction.model_validate_json(...)`, retrying (up to a few attempts) if the
   model's output doesn't parse.
3. Run it on all 6 tickets and compute: `category_accuracy`, `urgency_accuracy`, and
   `exact_match_accuracy` (both fields correct).

**Do not change:** `TICKETS`, the two field names of `TicketExtraction`, or the function
name `extract_ticket_info`.


In [9]:
# Fixed tickets with ground truth — DO NOT MODIFY
# Selects 6 specific real English tickets (by exact subject line) from the dataset, so
# every student works on the same tickets with the same true_category / true_urgency,
# taken directly from the dataset's own "queue" and "priority" columns.
FIXED_TICKET_SUBJECTS = [
    "Account Disruption",
    "Inquiry Regarding Invoice Details",
    "Connectivity Problems with Printer on MacBook Pro",
    "Problem with Payment Billing Procedures",
    "Audio Hardware Detected",
    "Query About Smart Home System Integration Features",
]

tickets_ds = load_dataset("Tobi-Bueck/customer-support-tickets")["train"]

_selected_by_subject = {}
for row in tickets_ds:
    if row["subject"] in FIXED_TICKET_SUBJECTS and row["subject"] not in _selected_by_subject:
        _selected_by_subject[row["subject"]] = row
    if len(_selected_by_subject) == len(FIXED_TICKET_SUBJECTS):
        break

TICKETS = [
    {
        "subject": subject,
        "text": _selected_by_subject[subject]["body"],
        "true_category": _selected_by_subject[subject]["queue"],
        "true_urgency": _selected_by_subject[subject]["priority"],
    }
    for subject in FIXED_TICKET_SUBJECTS
]

for t in TICKETS:
    print(f"[{t['true_category']} / {t['true_urgency']}] {t['subject']}")
    print(f"  {t['text'][:150]}...\n")


README.md:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

aa_dataset-tickets-multi-lang-5-2-50-ver(…): reconstructing file:   0%|          |  0.00B / 26.0MB            

aa_dataset-tickets-multi-lang-5-2-50-ver(…): downloading bytes:           |  0.00B            

(…)set-tickets-german_normalized_50_5_2.csv:   0%|          | 0.00/8.33M [00:00<?, ?B/s]

dataset-tickets-multi-lang-4-20k.csv: reconstructing file:   0%|          |  0.00B / 18.8MB            

dataset-tickets-multi-lang-4-20k.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/61765 [00:00<?, ? examples/s]

[Technical Support / high] Account Disruption
  Dear Customer Support Team,\n\nI am writing to report a significant problem with the centralized account management portal, which currently appears to...

[Billing and Payments / low] Inquiry Regarding Invoice Details
  Dear Customer Support Team,\n\nI hope this message finds you well. I am reaching out to request clarification about the billing and payment procedures...

[Technical Support / medium] Connectivity Problems with Printer on MacBook Pro
  Dear Support Team,\n\nI am reporting a recurring issue with the Laser Printer when printing from MacBook Pros running macOS 15. Several team members h...

[Billing and Payments / medium] Problem with Payment Billing Procedures
  Customer Service, I am reaching out to report an issue with the billing payment process on my account. Recently, there have been inconsistencies in th...

[Technical Support / low] Audio Hardware Detected
  Dear Customer Support Team,\n\nI am reaching out to request 

### TODO 2.1 — `TicketExtraction` Pydantic model (5 marks)

In [10]:
class TicketExtraction(BaseModel):
    category: Literal[
        "Technical Support",
        "Billing and Payments",
        "Returns and Exchanges",
    ]
    urgency: Literal["high", "medium", "low"]

# Quick check — should print the JSON schema
print(TicketExtraction.model_json_schema())

{'properties': {'category': {'enum': ['Technical Support', 'Billing and Payments', 'Returns and Exchanges'], 'title': 'Category', 'type': 'string'}, 'urgency': {'enum': ['high', 'medium', 'low'], 'title': 'Urgency', 'type': 'string'}}, 'required': ['category', 'urgency'], 'title': 'TicketExtraction', 'type': 'object'}


### TODO 2.2 — `extract_ticket_info` (7 marks)


In [11]:
def extract_ticket_info(ticket_text, max_attempts=3):
    """Prompt the local model to return JSON matching the TicketExtraction schema,
    and validate/retry until it parses (or raise after max_attempts).

    Unlike OpenAI's tool calling, open models have no built-in structural guarantee,
    so we ask for JSON explicitly and validate the result ourselves."""
    schema = TicketExtraction.model_json_schema()

    prompt = f"""Read the customer support ticket below and extract exactly two fields:
category and urgency.

Return ONLY one valid JSON object matching this schema:
{json.dumps(schema, indent=2)}

Do not include markdown, explanations, or any text outside the JSON object.

Ticket:
{ticket_text}
"""

    last_error = None

    for attempt in range(max_attempts):
        response = get_model_response(prompt, max_new_tokens=60)

        start = response.find("{")
        end = response.rfind("}")

        if start == -1 or end == -1 or end <= start:
            last_error = ValueError("The model response did not contain a JSON object.")
            continue

        json_text = response[start:end + 1]

        try:
            return TicketExtraction.model_validate_json(json_text)
        except Exception as exc:
            last_error = exc

    raise ValueError(
        f"Could not extract a valid TicketExtraction after {max_attempts} attempts. "
        f"Last error: {last_error}"
    )

# Quick check on the first ticket
print(extract_ticket_info(TICKETS[0]["text"]))

category='Technical Support' urgency='high'


### TODO 2.3 — Run extraction on all tickets and compute accuracy (5 marks)

In [12]:
# Run extraction on every ticket in the fixed order.
extraction_results = [
    extract_ticket_info(ticket["text"])
    for ticket in TICKETS
]

# Compute the three requested accuracy measures.
total_tickets = len(TICKETS)

category_accuracy = sum(
    result.category == ticket["true_category"]
    for result, ticket in zip(extraction_results, TICKETS)
) / total_tickets

urgency_accuracy = sum(
    result.urgency == ticket["true_urgency"]
    for result, ticket in zip(extraction_results, TICKETS)
) / total_tickets

exact_match_accuracy = sum(
    result.category == ticket["true_category"]
    and result.urgency == ticket["true_urgency"]
    for result, ticket in zip(extraction_results, TICKETS)
) / total_tickets

print(f"Category accuracy:    {category_accuracy}")
print(f"Urgency accuracy:     {urgency_accuracy}")
print(f"Exact-match accuracy: {exact_match_accuracy}")

Category accuracy:    0.8333333333333334
Urgency accuracy:     0.3333333333333333
Exact-match accuracy: 0.3333333333333333


## Section 3 — Evaluation Metrics: ROUGE, BLEU, Cohen's Kappa (16 marks)

**Part A — Dataset.** [CNN/DailyMail 3.0.0](https://huggingface.co/datasets/abisee/cnn_dailymail)
— the summarization dataset. For each of the first 3 articles of
the **test** split, `PAIRS` (built for you below) pairs:
- the dataset's own human-written summary (`highlights`) as the **reference**, with
- a simple, fully deterministic **extractive baseline candidate** — the first 2 sentences
  of the article — so nothing here depends on an LLM call.

For each pair, compute:
- ROUGE-1 and ROUGE-L F-measure, using `rouge_score.rouge_scorer` with `use_stemmer=True`.
- BLEU score, using `nltk.translate.bleu_score.sentence_bleu` on word-tokenized text.

**Part B.** You are given a fixed 3×3 confusion matrix (`CONFUSION_MATRIX`) of two
annotators labeling 12 movie-review sentences as Positive / Neutral / Negative. Compute Cohen's Kappa **by
hand** (do not use a library) using:

```
Po = (sum of diagonal cells) / N
Pe = sum over each category of (row_total * col_total) / N^2
kappa = (Po - Pe) / (1 - Pe)
```

**Do not change:** `PAIRS`, `CONFUSION_MATRIX`, or the function names below. No API key
is needed for this section.


In [13]:
# Fixed reference/candidate pairs — DO NOT MODIFY
# Reference = the dataset's own human summary. Candidate = a deterministic, non-LLM
# extractive baseline (first 2 sentences of the article), so Part A needs no API key.
cnn_dm = load_dataset("abisee/cnn_dailymail", "3.0.0")

def first_n_sentences(text, n=2):
    """Return the first n sentences of text (naive split on '. '), as a simple
    extractive baseline summary."""
    cleaned = text.replace("\n", " ").strip()
    sentences = cleaned.split(". ")
    summary = ". ".join(sentences[:n]).strip()
    if not summary.endswith("."):
        summary += "."
    return summary

PAIRS = []
for row in cnn_dm["test"].select(range(3)):
    reference = row["highlights"].replace("\n", " ").strip()
    candidate = first_n_sentences(row["article"], n=2)
    PAIRS.append({"id": row["id"], "reference": reference, "candidate": candidate})

for pair in PAIRS:
    print(f"ID: {pair['id']}")
    print(f"Reference: {pair['reference']}")
    print(f"Candidate (naive baseline): {pair['candidate']}\n")

# Fixed confusion matrix — DO NOT MODIFY
# Rows = Annotator A, Columns = Annotator B, order = [Positive, Neutral, Negative]
CONFUSION_MATRIX = {
    "labels": ["Positive", "Neutral", "Negative"],
    "matrix": [
        [4, 1, 0],
        [0, 2, 1],
        [0, 1, 3],
    ],
}


README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

3.0.0/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

...

3.0.0/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

ID: f001ec5c4704938247d27a44948eebb37ae98d01
Reference: Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June . Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .
Candidate (naive baseline): (CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based.

ID: 230c522854991d053fe98a718b1defa077a8efef
Reference: Theia, a bully breed mix, was apparently hit by a car, whacked with a hammer and buried in a field . "She's a true miracle dog and she deserves a good life," says Sara Mellado, who is looking for a home for Theia .
Candidate (naive baseline): (CNN)Never mind cats having nine lives. A stray pooch in Washington S

### TODO 3.1 — `compute_rouge_scores` and `compute_bleu_score` (6 marks)

In [14]:
def compute_rouge_scores(reference, candidate):
    """Return {"rouge1_f": ..., "rougeL_f": ...} rounded to 4 decimals."""
    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rougeL"],
        use_stemmer=True,
    )
    scores = scorer.score(reference, candidate)

    return {
        "rouge1_f": round(scores["rouge1"].fmeasure, 4),
        "rougeL_f": round(scores["rougeL"].fmeasure, 4),
    }


def compute_bleu_score(reference, candidate):
    """Return the sentence-level BLEU score rounded to 4 decimals."""
    reference_tokens = word_tokenize(reference)
    candidate_tokens = word_tokenize(candidate)

    score = sentence_bleu(
        [reference_tokens],
        candidate_tokens,
    )

    return round(score, 4)

### TODO 3.2 — Run metrics over all pairs (4 marks)

In [15]:
# Compute the requested metrics for every fixed reference/candidate pair.
metrics_results = []

for pair in PAIRS:
    rouge_results = compute_rouge_scores(
        pair["reference"],
        pair["candidate"],
    )
    bleu = compute_bleu_score(
        pair["reference"],
        pair["candidate"],
    )

    metrics_results.append({
        "rouge1_f": rouge_results["rouge1_f"],
        "rougeL_f": rouge_results["rougeL_f"],
        "bleu": bleu,
    })

for pair, result in zip(PAIRS, metrics_results):
    print(f"Reference: {pair['reference']}")
    print(f"Candidate: {pair['candidate']}")
    print(
        f"  ROUGE-1 F: {result['rouge1_f']}  "
        f"ROUGE-L F: {result['rougeL_f']}  "
        f"BLEU: {result['bleu']}\n"
    )

Reference: Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June . Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .
Candidate: (CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based.
  ROUGE-1 F: 0.2927  ROUGE-L F: 0.2927  BLEU: 0.0758

Reference: Theia, a bully breed mix, was apparently hit by a car, whacked with a hammer and buried in a field . "She's a true miracle dog and she deserves a good life," says Sara Mellado, who is looking for a home for Theia .
Candidate: (CNN)Never mind cats having nine lives. A stray pooch in Washington State has used up at least three of her own after being hit by a car, ap

/usr/local/lib/python3.13/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


### TODO 3.3 — `compute_cohens_kappa` (6 marks)

In [16]:
def compute_cohens_kappa(matrix):
    """Given a square confusion matrix (list of lists of ints), return
    (kappa, po, pe) all rounded to 4 decimals."""
    n = sum(sum(row) for row in matrix)

    # Observed agreement.
    diagonal_total = sum(matrix[i][i] for i in range(len(matrix)))
    po = diagonal_total / n

    # Row and column totals.
    row_totals = [sum(row) for row in matrix]
    col_totals = [
        sum(matrix[row][col] for row in range(len(matrix)))
        for col in range(len(matrix[0]))
    ]

    # Expected agreement by chance.
    pe = sum(
        row_total * col_total
        for row_total, col_total in zip(row_totals, col_totals)
    ) / (n ** 2)

    kappa = (po - pe) / (1 - pe)

    return round(kappa, 4), round(po, 4), round(pe, 4)


kappa_value, po_value, pe_value = compute_cohens_kappa(CONFUSION_MATRIX["matrix"])
print(f"Observed agreement (Po): {po_value}")
print(f"Expected agreement (Pe): {pe_value}")
print(f"Cohen's Kappa: {kappa_value}")

Observed agreement (Po): 0.75
Expected agreement (Pe): 0.3333
Cohen's Kappa: 0.625
